# Cognitive Profile EDA

Exploratory analysis of cognitive dimension scores from the assessment pipeline.

In [ ]:
import sys; sys.path.insert(0, '..')
import asyncio, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from src.metrics.visualizations import run_full_report

# Generate all report figures
run_full_report(
    db_path=Path('../data/chat_logs/sessions.db'),
    output_dir=Path('../data/reports')
)
print('Report saved to ../data/reports/')

In [ ]:
# Load scores into pandas
import sqlite3
conn = sqlite3.connect('../data/chat_logs/sessions.db')
df = pd.read_sql('SELECT * FROM scores', conn)
print(df.shape)
df.head()

In [ ]:
# ICAP level distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['dominant_icap'].value_counts().plot(kind='bar', ax=axes[0], color=['#ef4444','#f97316','#22c55e','#3b82f6'])
axes[0].set_title('ICAP Level Distribution'); axes[0].set_xlabel('')

# Duration distribution by ICAP
for level, color in [('Passive','#ef4444'),('Active','#f97316'),('Constructive','#22c55e'),('Interactive','#3b82f6')]:
    subset = df[df['dominant_icap']==level]['duration_sec'].dropna()
    if len(subset): axes[1].hist(subset, bins=15, alpha=0.6, label=level, color=color)
axes[1].set_title('Response Duration by ICAP Level'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap between dimensions
dims = ['C1_dominant','C2_dominant','C3_dominant','C4_dominant',
        'C5_dominant','C6_dominant','C7_dominant','C8_dominant']
numeric_df = df[dims].apply(pd.to_numeric, errors='coerce')
corr = numeric_df.corr(method='spearman')
plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=[d.replace('_dominant','') for d in dims],
            yticklabels=[d.replace('_dominant','') for d in dims])
plt.title('Spearman Correlation: Cognitive Dimensions')
plt.tight_layout(); plt.show()